In [ ]:
# Notebooks live in notebooks/; make the repo root importable (ppo, train_ppo, ...).
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "ppo" / "paths.py").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))


# PPO Workbench

One notebook for the full PPO workflow: **train → monitor → evaluate → plot → compare**.
Backed by the `ppo/` package — see [PPO_PIPELINE.md](PPO_PIPELINE.md) for architecture and artifact reference.

Tips:
- `backend="analytic"` runs everything MATLAB-free in seconds — use it to iterate on configs/plots.
- Repeated MATLAB evaluations in this kernel reuse **one** engine (`get_shared_session`). Run `matlab.engine.shareEngine` in a MATLAB console first and attaching is instant.
- CLI equivalents exist for everything: `python -m ppo --help`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ppo import (
    ExperimentConfig, BandConfig, RewardConfig, RewardWeights,
    train, evaluate_run, list_runs, load_run,
    get_shared_session, close_shared_session,
)
from ppo import plotting

## 1. Pipeline smoke (no MATLAB, ~10 s)

Sanity-check the whole pipeline after any code change — or run `python -m ppo smoke` in a terminal.

In [ ]:
exp_smoke = ExperimentConfig.from_code(
    "1-1-1",
    band=BandConfig(mode="multi", fbs_band="agent", mbs_capacity="agent"),
    tag="nb_smoke",
    total_timesteps=1024,
    seed=0,
    ppo_overrides={"n_steps": 64, "batch_size": 64, "verbose": 0},
    world_overrides={"num_users": 300},
)
smoke_dir = train(exp_smoke, backend="analytic")
smoke_report = evaluate_run(smoke_dir, episodes=2, backend="analytic")
smoke_report.summary_df

## 2. Real training (MATLAB)

Same call with `backend="matlab"` (default). Band modes:
- `BandConfig()` → legacy single-band world (old agents comparable)
- `BandConfig(mode="multi", fbs_band="agent", mbs_capacity="agent")` → dual-band world
- `reward=RewardConfig(mode="controlled_blend", ...)` → bills only agent-controlled power

In [ ]:
exp = ExperimentConfig.from_code(
    "1-1-1",                       # X-Y-Z: FBS count - scenario - cost config
    # band=BandConfig(mode="multi", fbs_band="agent", mbs_capacity="agent"),
    # reward=RewardConfig(mode="controlled_blend", weights=RewardWeights(beta=0.8, gamma=0.0, fbs_weight=0.4)),
    total_timesteps=3_000,
    max_episode_steps=30,
    ent_coef=0.7,
    action_scale=0.9,
    learning_rate=1e-4,
    seed=0,
    # n_envs=4,                    # opt-in parallel rollouts (one engine per worker)
    ppo_overrides={"checkpoint_every": 1_000, "eval_every": 1_000},
    tag=None,
)

# run_dir = train(exp)             # uncomment to launch a real run
# run_dir

## 3. Runs on disk

All generations are listed together — new (`v2`), old harness (`v1`), and bare notebook-era runs.

In [ ]:
list_runs().tail(15)

In [ ]:
run_dir = "latest"           # or e.g. "run_2026-07-04_13-57-15_1-1-1_mb_smoke" / "run_039"
handle = load_run(run_dir)
print(handle.name, "|", handle.schema, "|", handle.model_path.name if handle.model_path else None)

## 4. Training curves

Everything renders from the run's flat CSVs (`steps.csv`, `episodes.csv`, `progress.csv`).

In [ ]:
fig = plotting.plot_training_overview(handle.run_dir, window=20)

In [ ]:
fig = plotting.plot_reward_components(handle.run_dir)

In [ ]:
# Raw frames, if you want custom analysis
frames = plotting.load_training_frames(handle.run_dir)
frames["episodes"].tail() if frames["episodes"] is not None else None

## 5. Evaluate a policy

Seeded episodes → identical start states every time you evaluate (comparable numbers).
Artifacts + a full plot gallery land under `<run_dir>/evals/<timestamp>/`.

Works on any run generation — for bare old runs the env is reconstructed from the model's observation space.

In [ ]:
report = evaluate_run(
    handle.run_dir,
    episodes=3,
    deterministic=True,
    # initial_state=np.array([800, 800, 100, 10.5, 1], dtype=np.float32),  # pin the start
    # max_episode_steps=100,
    # backend="analytic",          # quick mechanical check without MATLAB
    # mirror_test_logs=True,       # also write old test_logs/<code>/ files
)
report.summary_df

In [ ]:
report.aggregate

### Inline gallery for one episode

In [ ]:
ep = report.episodes[0]
fig = plotting.plot_trajectory_map(
    ep.state_df, report.mbs_x, report.mbs_y,
    world=report.exp.env.world, users=ep.user_positions,
    title=f"{report.run_dir.name} — episode {ep.index}",
)

In [ ]:
fig = plotting.plot_step_metrics(ep.metrics_df)
fig = plotting.plot_altitude_power(ep.state_df)

In [ ]:
if "mbs_capacity_connected" in ep.metrics_df:
    fig = plotting.plot_connectivity_stack(ep.metrics_df)

## 6. Compare runs

In [ ]:
runs_to_compare = list_runs().query("schema == 'v2'")["run"].tail(3).tolist()
if len(runs_to_compare) > 1:
    from ppo import resolve_run
    fig = plotting.plot_learning_curves(
        [resolve_run(r) for r in runs_to_compare], metric="reward_sum", window=20,
    )

## 7. Trajectory overlay

`plot_trajectories.py` reads evaluation trajectory CSVs directly. Point it at
one or more eval trajectories; give them different *groups* to contrast two
policies with distinct colors and linestyles:

In [ ]:
# from plot_trajectories import plot_trajectories
# plot_trajectories([
#     (str(report_a.eval_dir / "ep00_trajectory.csv"), "baseline", "a"),
#     (str(report_b.eval_dir / "ep00_trajectory.csv"), "shaped",   "b"),
# ], title="Rollout comparison")

## 8. Housekeeping

In [ ]:
# close_shared_session()   # release the shared MATLAB engine when done